In [0]:
-- ============================================================
-- Script : 04_load_dimensions_korrigiert
-- Purpose: Load all warehouse dimension tables from staging.
--          dim_customer uses SCD Type 2 (loyalty_tier tracked historically);
--          dim_product, dim_store are full overwrites; dim_date is a
--          generated calendar dimension (2020-2030).
-- Depends: brewflow_staging (stg_customers, stg_products, stg_stores),
--          brewflow_warehouse (dim_customer, dim_product, dim_store, dim_date)
-- Outputs: brewflow_warehouse (dim_customer, dim_product, dim_store, dim_date)
-- ============================================================

-- dim_customer: SCD Type 2 MERGE on loyalty_tier
-- Step 1: close the current row if loyalty_tier changed
MERGE INTO brewflow_warehouse.dim_customer AS target
USING (
    SELECT customer_id, first_name, last_name, email, city, loyalty_tier
    FROM brewflow_staging.stg_customers
) AS source
ON target.customer_id = source.customer_id AND target.is_current = TRUE
WHEN MATCHED AND target.loyalty_tier <> source.loyalty_tier THEN
    UPDATE SET effective_to = date_sub(current_date(), 1), is_current = FALSE;

-- Step 2: insert a new current row for changed customers (after closing the old one)
INSERT INTO brewflow_warehouse.dim_customer
    (customer_id, first_name, last_name, email, city, loyalty_tier,
     effective_from, effective_to, is_current)
SELECT src.customer_id, src.first_name, src.last_name, src.email,
       src.city, src.loyalty_tier, current_date(), NULL, TRUE
FROM brewflow_staging.stg_customers src
JOIN brewflow_warehouse.dim_customer dc
    ON src.customer_id = dc.customer_id
    AND dc.is_current   = FALSE
    AND dc.effective_to = date_sub(current_date(), 1)
LEFT JOIN brewflow_warehouse.dim_customer current_row
    ON src.customer_id = current_row.customer_id AND current_row.is_current = TRUE
WHERE current_row.customer_sk IS NULL;

-- Step 3: insert brand-new customers that don't exist in dim_customer at all
INSERT INTO brewflow_warehouse.dim_customer
    (customer_id, first_name, last_name, email, city, loyalty_tier,
     effective_from, effective_to, is_current)
SELECT src.customer_id, src.first_name, src.last_name, src.email,
       src.city, src.loyalty_tier, current_date(), NULL, TRUE
FROM brewflow_staging.stg_customers src
WHERE NOT EXISTS (
    SELECT 1 FROM brewflow_warehouse.dim_customer dc
    WHERE dc.customer_id = src.customer_id
);

-- dim_product: full overwrite (explizite Spaltenliste, da product_sk Identity-Spalte ist)
INSERT OVERWRITE brewflow_warehouse.dim_product (product_id, name, category, unit_price)
    SELECT product_id, name, category, unit_price FROM brewflow_staging.stg_products;

-- dim_store: full overwrite (explizite Spaltenliste, da store_sk Identity-Spalte ist)
INSERT OVERWRITE brewflow_warehouse.dim_store (store_id, name, city, country)
    SELECT store_id, name, city, country FROM brewflow_staging.stg_stores;

-- dim_date: generate calendar 2020-2030 (idempotent, only inserts missing days)
INSERT INTO brewflow_warehouse.dim_date
    (date_sk, calendar_date, year, quarter, month, month_name, day_of_week, is_weekend)
SELECT
    CAST(date_format(cal, 'yyyyMMdd') AS INT),
    cal,
    year(cal),
    quarter(cal),
    month(cal),
    date_format(cal, 'MMMM'),
    date_format(cal, 'EEEE'),
    dayofweek(cal) IN (1, 7)
FROM (
    SELECT explode(sequence(DATE'2020-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS cal
) dates
WHERE NOT EXISTS (
    SELECT 1 FROM brewflow_warehouse.dim_date d
    WHERE d.date_sk = CAST(date_format(cal, 'yyyyMMdd') AS INT)
);

-- Validieren
SELECT 'dim_customer' AS tbl, COUNT(*) AS rows FROM brewflow_warehouse.dim_customer
UNION ALL SELECT 'dim_product', COUNT(*) FROM brewflow_warehouse.dim_product
UNION ALL SELECT 'dim_store',   COUNT(*) FROM brewflow_warehouse.dim_store
UNION ALL SELECT 'dim_date',    COUNT(*) FROM brewflow_warehouse.dim_date;
-- Erwartet: dim_customer >= 10, dim_product 8, dim_store 5, dim_date ca. 4018